# **Overview**

This project develops a Hybrid AI Chatbot that combines a Large Language Model (Microsoft Phi-2) with a custom product recommendation system. The chatbot can handle general conversations using an LLM while also providing intelligent product recommendations based on user requirements.

The system uses intent detection to determine whether a user query should be processed by the conversational AI module or the product recommendation engine. It also maintains conversational memory to support follow-up questions and provide context-aware responses.

The product recommendation module analyzes factors such as price, ratings, competition, customer reviews, and profit margins to rank products and generate relevant suggestions. This project demonstrates the integration of NLP, conversational AI, recommendation systems, and autonomous decision-making within a single intelligent application.


In [ ]:
# ===========================================
# INSTALL DEPENDENCIES
# ===========================================
!pip install -q transformers accelerate sentencepiece torch

import os
import re
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

# ===========================================
# LOAD LLM (Phi-2)
# ===========================================
MODEL_NAME = "microsoft/phi-2"

print("🔄 Loading Phi-2 model...")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device set to: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)


# ===========================================
# HELPER: BUILD CHAT PROMPT & REPLY
# ===========================================
def build_prompt(history, user_message: str) -> str:
    """
    Simple prompt format for Phi-2 with a clear system instruction.
    """
    system_msg = (
        "You are a helpful, professional AI assistant. "
        "Answer clearly and directly. Do not roleplay or write fictional emails "
        "unless the user explicitly asks.\n\n"
    )

    prompt = system_msg
    for turn in history:
        prompt += f"User: {turn['user']}\nAssistant: {turn['bot']}\n"
    prompt += f"User: {user_message}\nAssistant:"
    return prompt


def llm_reply(user_message: str, history, max_new_tokens=180) -> str:
    prompt = build_prompt(history, user_message)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.6,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    text = tokenizer.decode(output[0], skip_special_tokens=True)
    reply = text[len(prompt):].strip()

    # Clean any extra "User:" / "Assistant:" that model might add
    reply = reply.split("User:")[0].strip()
    reply = reply.split("Assistant:")[0].strip()
    return reply


# ===========================================
# LOAD PRODUCTS CSV
# ===========================================
CSV_PATH = "products.csv"

def load_products(path=CSV_PATH):
    if not os.path.exists(path):
        print("❌ ERROR: 'products.csv' not found. Upload it in the Colab Files panel.")
        return pd.DataFrame()

    df = pd.read_csv(path)

    numeric = [
        "price","rating","review_count","bsr","competition",
        "cost_price","shipping_cost","referral_fee_rate"
    ]
    for col in numeric:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["category"] = df["category"].astype(str).str.lower()
    df["features"] = df["features"].astype(str).str.lower()
    df["product_name"] = df["product_name"].astype(str)
    return df

products_df = load_products()
if products_df.empty:
    KNOWN_CATEGORIES = []
else:
    KNOWN_CATEGORIES = sorted(products_df["category"].unique().tolist())


# ===========================================
# INTENT DETECTION & QUERY PARSING
# ===========================================
SEARCH_WORDS = [
    "find","search","recommend","suggest","product","products",
    "item","items","under","around","budget","price","buy","show"
]

STOP_WORDS = {
    "i","need","a","an","the","with","and","for","find","search",
    "under","around","budget","please","show","me","some","any",
    "good","best","about","give","tell","hows","how","is","are",
    "it","like","of","do","you","think","on","this","that","one",
    "camera","router","access","point","ap"
}

def is_product_intent(text: str) -> bool:
    t = text.lower()
    if any(cat in t for cat in KNOWN_CATEGORIES):
        return True
    if any(w in t for w in SEARCH_WORDS):
        return True
    if re.search(r"\$\s*\d+|\d+\s*(dollars|\$)", t):
        return True
    return False


def parse_query(text: str):
    t = text.lower()

    # Price detection
    nums = re.findall(r"\d+\.?\d*", t)
    price = float(nums[0]) if nums else None

    # Category detection
    category = None
    for c in KNOWN_CATEGORIES:
        if c in t.replace(" ", "_"):
            category = c
            break

    # Feature keywords
    words = re.findall(r"[a-zA-Z0-9]+", t)
    features = [
        w.lower() for w in words
        if w.lower() not in STOP_WORDS
        and w.lower() not in KNOWN_CATEGORIES
        and not w.isdigit()
    ]

    return {"price": price, "category": category, "features": features}


# ===========================================
# PRODUCT SCORING
# ===========================================
def score_products(df: pd.DataFrame, query: dict) -> pd.DataFrame:
    if df.empty:
        return df

    price  = query["price"]
    cat    = query["category"]
    feats  = query["features"]

    scores = []
    for _, row in df.iterrows():
        s = 0

        # Category
        if cat:
            s += 7 if row["category"] == cat else -2

        # Feature match
        if feats:
            matches = sum(1 for f in feats if f in row["features"])
            s += matches * 2
            if matches == 0:
                s -= 1

        # Price closeness
        if price is not None:
            diff = abs(row["price"] - price)
            if diff < 5: s += 10
            elif diff < 15: s += 6
            elif diff < 40: s += 4
            elif diff < 80: s += 2
            else: s -= 4

        # Demand factors
        s += row["rating"] * 2
        s += min(row["review_count"] / 500, 10)  # cap contribution
        s += max(0, (5000 - row["bsr"]) / 800)   # lower BSR = better

        # Competition (lower better)
        s += max(0, (50 - row["competition"]) / 5)

        # Profit margin
        fee = row["price"] * row["referral_fee_rate"]
        margin = row["price"] - (row["cost_price"] + row["shipping_cost"] + fee)
        if margin > 50: s += 4
        elif margin > 20: s += 2

        scores.append(s)

    out = df.copy()
    out["score"] = scores
    out = out[out["score"] > 0]
    return out.sort_values("score", ascending=False)


def format_row(row) -> str:
    fee = row["price"] * row["referral_fee_rate"]
    margin = row["price"] - (row["cost_price"] + row["shipping_cost"] + fee)
    return (
        f"- {row['product_name']} (${row['price']:.2f})\n"
        f"  • Category: {row['category']}\n"
        f"  • Rating: {row['rating']:.1f} ⭐ ({int(row['review_count'])} reviews)\n"
        f"  • Profit Margin: ${margin:.2f}\n"
        f"  • Features: {row['features']}\n"
    )


# ===========================================
# PRODUCT ENGINE WITH FOLLOW-UP MEMORY
# ===========================================
def product_answer(user_text: str, df: pd.DataFrame, last_results: pd.DataFrame):
    """
    - If user refers to a previously shown product (by full name or brand),
      show only that product.
    - Otherwise, run a fresh scored search.
    """
    if df.empty:
        return "I don't have access to the product dataset. Please upload products.csv.", None

    t = user_text.lower()

    # 1) Follow-up: look in last_results for product name or brand
    if last_results is not None and not last_results.empty:
        for _, row in last_results.iterrows():
            product_name = row["product_name"].lower()

            # full product name mentioned
            if product_name in t:
                return format_row(row), last_results

            # brand detection (first word of product name)
            brand = product_name.split()[0]              # e.g. "Reolink", "Ubiquiti"
            brand_clean = re.sub(r"[^a-zA-Z0-9]", "", brand).lower()

            if len(brand_clean) > 2 and brand_clean in t:
                return format_row(row), last_results

    # 2) Fresh search
    q = parse_query(user_text)
    results = score_products(df, q)

    if results.empty:
        return (
            "I couldn't find strong matches for that request. "
            "Try changing your budget, category, or removing some constraints."
        ), None

    lines = ["Here are some product suggestions:\n"]
    for _, row in results.head(5).iterrows():
        lines.append(format_row(row))
    lines.append("You can refine by changing your budget, category, or adding more features.")

    # store top 20 in memory for follow-ups
    return "\n".join(lines), results.head(20)


# ===========================================
# MAIN CHAT LOOP
# ===========================================
def chat():
    print("========================================")
    print("   HYBRID CHATBOT (Phi-2 + Product AI)")
    print("========================================")
    print("• Chat normally (Phi-2 handles general questions).")
    print("• Ask things like 'recommend a router around $200' for product mode.")
    print("• Use follow-ups like 'what about reolink?' to see details.")
    print("• Type 'exit' or 'quit' to stop.\n")

    history = []       # for LLM context
    last_results = None

    while True:
        user = input("You: ").strip()
        if not user:
            continue

        if user.lower() in ["exit", "quit"]:
            print("Bot: Goodbye!")
            break

        # Decide which brain to use
        if is_product_intent(user):
            reply, last_results = product_answer(user, products_df, last_results)
            print("\nBot (Product AI):")
            print(reply + "\n")
            history.append({"user": user, "bot": reply})
        else:
            reply = llm_reply(user, history)
            print(f"\nBot: {reply}\n")
            history.append({"user": user, "bot": reply})


# ===========================================
# RUN
# ===========================================
chat()


🔄 Loading Phi-2 model...
Device set to: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

   HYBRID CHATBOT (Phi-2 + Product AI)
• Chat normally (Phi-2 handles general questions).
• Ask things like 'recommend a router around $200' for product mode.
• Use follow-ups like 'what about reolink?' to see details.
• Type 'exit' or 'quit' to stop.


Bot: I'm fine, thank you for asking. How can I assist you today?


Bot: A neural network is a type of machine learning algorithm that is modeled after the structure and function of the human brain. It consists of interconnected nodes, called neurons, which are organized into layers. The input layer receives data, the hidden layers perform computations, and the output layer produces the final result.

Each neuron takes inputs from other neurons, applies an activation function to it, and produces an output. The activation function determines whether the neuron should be activated or not based on the input it receives. The weights and biases of the neurons are adjusted during training to minimize the difference between the predicted output